# Calculation of the Convex Hull of the Fe-Cu-Co-Ni System
Data source: Materials Project

In [1]:
import numpy as np
import pandas as pd
from convexhull import cvhull

In [2]:
help(cvhull)

Help on class cvhull in module convexhull:

class cvhull(builtins.object)
 |  cvhull(name, composition, energy, species=None, accuracy=4)
 |  
 |  Class to calculate convexhull for materials data.
 |  
 |  
 |  Parameters :
 |  ------------
 |  * name (list, len=N) : Names of the compounds
 |  * composition (np.array([N,c], float)) : Compositions of the compounds (or number of atoms)
 |  * energy (np.array([N], float)) : Energy of the phases
 |  * species (list(str), len=c) : List of chemical species (default: A, B, C, ...)
 |  * accuracy (int) : Number of significant digits used for the compositions (default: 4)
 |  
 |  N: number of compound
 |  c: number of species
 |  
 |  Attributs :
 |  -----------
 |  * name (list, len=N) : Name of compounds
 |  * composition (np.array([N,c], float) : Composition of compounds
 |  * energy (np.array([N], float) : Energy of the phases
 |  * species (list, len=c) : list of chemical species (default: A,B,C,...)
 |  * npoints (int) : Compounds number

## Input data
* name (list, len=N) : Names of the compounds
* composition (np.array([N,c], float)) : Compositions of the compounds (or number of atoms)
* energy (np.array([N], float)) : Energies of the phases
* species (list(str), len=c) : List of chemical species (default: A, B, C, ...)
* accuracy (int) : Number of significant digits used for the compositions (default: 4)

In [3]:
df = pd.read_csv("data/incomplet.csv")
df

,name,A,B,C,D,sum,energy
0,Fe,1,0,0,0,1,-8.19
1,Co,0,0,1,0,1,-7.11
2,Ni,0,0,0,1,1,-5.37
3,CuNi,0,1,0,1,2,-4.91
4,FeCuNi,1,1,0,1,3,-6.05
5,CoNi,0,0,1,1,2,-7.00
6,CoNi2,0,0,1,1,2,-6.50


In [4]:
name = df.index
composition = df.loc[:,['A','B','C','D']]
energy = df['energy']
try :
    cvhull(name , composition, energy)
except Exception as e:
    print("Error:",e)

Error: All reference (purs elements) must be in database


**Warning:** there is one constraint: all reference phases must be present in the database.

In [5]:
df = pd.read_csv("data/data.csv")
df

,id,formula,Fe,Cu,Co,Ni,sum,E
0,mp-1245108,Fe,1,0,0,0,1,-8.19
1,mp-1271198,Fe,1,0,0,0,1,-8.42
2,mp-13,Fe,1,0,0,0,1,-8.47
3,mp-1194030,Fe,1,0,0,0,1,-8.29
4,mp-1271068,Fe,1,0,0,0,1,-8.45
...,...,...,...,...,...,...,...,...
78,mp-1225698,Cu₃Ni,0,3,0,1,4,-4.47
79,mp-1225695,CuNi,0,1,0,1,2,-4.87
80,mp-1225687,CuNi,0,1,0,1,2,-4.92
81,mp-1184069,CuNi,0,1,0,1,2,-4.91


In [6]:
energy = df.loc[:,'E']

name = [x.strip() for x in df.loc[:,'id']]

element = ['Fe','Cu','Co','Ni']
composition = df.loc[:,element]

energy.shape, len(name), composition.shape
hull = cvhull(name, composition, energy, species=element)

## List of stable phases

In [7]:
hull.print_stables()

The database consist of 83 compounds with 10 stable

List of stable compounds:
-------------------------
-----------------------------------------------------------
  i   Name              Fe      Cu      Co      Ni   Energy
-----------------------------------------------------------
  2   mp-13         1.0000  0.0000  0.0000  0.0000    -8.47
 13   mp-54         0.0000  0.0000  1.0000  0.0000    -7.11
 21   mp-23         0.0000  0.0000  0.0000  1.0000    -5.78
 29   mp-30         0.0000  1.0000  0.0000  0.0000    -4.10
 35   mp-18695      0.9375  0.0000  0.0625  0.0000    -8.40
 38   mp-601820     0.7500  0.0000  0.2500  0.0000    -8.18
 40   mp-2090       0.5000  0.0000  0.5000  0.0000    -7.86
 55   mp-1418       0.2500  0.0000  0.0000  0.7500    -6.54
 58   mp-2213       0.5000  0.0000  0.0000  0.5000    -7.19
 68   mp-1183837    0.0000  0.0000  0.7500  0.2500    -6.80
-----------------------------------------------------------


## Output about one phase

In [8]:
MP_id = "mp-1225687"

### Energy

In [9]:
float(hull.energy_compound(MP_id))

-4.92

### Deviation from convex hull

In [10]:
dH = hull.distance_hull(MP_id, verbose=False)
print(f"DfH = {dH:.2f}")

DfH = 0.02


### Decomposition

In [11]:
dH = hull.distance_hull(MP_id)

The compound mp-1225687 is not stable
--------------------------------------------
ΔH = 0.02
Decomposition:  0.5(mp-23) + 0.5(mp-30) 


### All output

In [13]:
hull.info_compound('mp-2090') #If stable

Info about compound : mp-2090
------------------------------------
E = -7.86
The compound mp-2090 is stable!


In [15]:
hull.info_compound(MP_id) #If not

Info about compound : mp-1225687
------------------------------------
E = -4.92
The compound mp-1225687 is not stable
--------------------------------------------
ΔH = 0.02
Decomposition:  0.5(mp-23) + 0.5(mp-30) 


## Ouput for a given composition

In [16]:
compo = [0.2,0.5,0.2,0.1]

### Energy

In [18]:
float(hull.energy_hull(compo))

-5.7792

### Equilibrium

In [19]:
hull.equilibrium(compo)

Decomposition:  0.5(mp-30) + 0.34(mp-2090) + 0.12(mp-1418) + 0.04(mp-1183837) 


## Deviation from convex hull on all phases

In [20]:
dic = hull.table_dhull()
pd.DataFrame(dic)

,Name,Fe,Cu,Co,Ni,Stability,Deviation from hull
0,mp-1245108,1.000000,0.000000,0.0,0.000000,False,0.28
1,mp-1271198,1.000000,0.000000,0.0,0.000000,False,0.05
2,mp-13,1.000000,0.000000,0.0,0.000000,True,0.00
3,mp-1194030,1.000000,0.000000,0.0,0.000000,False,0.18
4,mp-1271068,1.000000,0.000000,0.0,0.000000,False,0.02
...,...,...,...,...,...,...,...
78,mp-1225698,0.000000,0.750000,0.0,0.250000,False,0.05
79,mp-1225695,0.000000,0.500000,0.0,0.500000,False,0.07
80,mp-1225687,0.000000,0.500000,0.0,0.500000,False,0.02
81,mp-1184069,0.000000,0.500000,0.0,0.500000,False,0.03
